In [1]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

In [2]:
# Check the GPU status
!nvidia-smi

Fri Nov 27 12:39:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Libraries

In [3]:
%%capture
!uv pip uninstall torchao torchaudio torchvision -y
!uv pip install \
    "transformers==4.53.3" \
    "peft==0.17.1" \
    "trl" \
    "accelerate" \
    "bitsandbytes" \
    "wandb"

In [4]:
from datetime import datetime
from transformers import AutoModelForQuestionAnswering, AutoTokenizer
from peft import PeftModel

# Configurations

In [5]:
# Run configuration
SRC_LANG = 'en'
TGT_LANG = 'tr'

# Model configuration
# MODEL_ID: str = 'FacebookAI/xlm-roberta-base'
# LORA_ID = 'alxxtexxr/XLM-R-Base-squad-en-6K-s42-LoRA-v261126181606'
# LORA_CKPT_STEP = 2835
# ---
# MODEL_ID: str = 'FacebookAI/xlm-roberta-base'
# LORA_ID = 'alxxtexxr/XLM-R-Base-squad-en-6K-s123-LoRA-v261126223748'
# LORA_CKPT_STEP = 2835
# LORA_CKPT_STEP = 3024
# ---
MODEL_ID: str = 'FacebookAI/xlm-roberta-base'
LORA_ID = 'alxxtexxr/XLM-R-Base-squad-en-6K-s1337-LoRA-v261127000059'
LORA_CKPT_STEP = 2835

# Set up the hub merged model ID
model_name = ""
if MODEL_ID == 'FacebookAI/xlm-roberta-base':
    model_name, _ = LORA_ID.split("-LoRA")
else:
    if SRC_LANG in MODEL_ID:
        model_id_0, model_id_1 = MODEL_ID.split(SRC_LANG)
    elif TGT_LANG in LORA_ID:
        model_id_0, model_id_1 = LORA_ID.split(TGT_LANG)
    model_id_1_0 = model_id_1.split("K-")[-1].split("-LoRA")[0]
    model_name = f"{model_id_0}{TGT_LANG}-{model_id_1_0}"
hub_merged_model_id = f"{model_name}-LoRA-a2m-ckpt{LORA_CKPT_STEP}-v{datetime.now().strftime('%y%m%d%H%M%S')}"
print(f"Hub merged model ID: {hub_merged_model_id}")

Hub merged model ID: alxxtexxr/XLM-R-Base-squad-en-6K-s1337-LoRA-a2m-ckpt2835-v261127123957


# Model

In [6]:
# Load the base model
base_model = AutoModelForQuestionAnswering.from_pretrained(MODEL_ID, device_map='auto')
print("device:", base_model.device)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
Some weights of XLMRobertaForQuestionAnswering were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


device: cuda:0


In [7]:
# Sanity check
for i in range(11):
    print(f"Base model layer-{i} attention value weight norm:", base_model.roberta.encoder.layer[i].attention.self.value.weight.norm().item())
print()
print("Base model qa_outputs weight norm:", base_model.qa_outputs.weight.norm().item())
print("Base model qa_outputs bias norm:", base_model.qa_outputs.bias.norm().item())

Base model layer-0 attention value weight norm: 28.364133834838867
Base model layer-1 attention value weight norm: 28.845458984375
Base model layer-2 attention value weight norm: 30.10150146484375
Base model layer-3 attention value weight norm: 36.34113693237305
Base model layer-4 attention value weight norm: 38.29389953613281
Base model layer-5 attention value weight norm: 41.62784957885742
Base model layer-6 attention value weight norm: 38.20062255859375
Base model layer-7 attention value weight norm: 38.956634521484375
Base model layer-8 attention value weight norm: 39.158668518066406
Base model layer-9 attention value weight norm: 34.89980697631836
Base model layer-10 attention value weight norm: 31.030277252197266

Base model qa_outputs weight norm: 0.7889131903648376
Base model qa_outputs bias norm: 0.0


In [8]:
# Load and merge the LoRA adapter into the base model
lora_model = PeftModel.from_pretrained(base_model, subfolder=f"checkpoint-{LORA_CKPT_STEP}", model_id=LORA_ID)
merged_model = lora_model.merge_and_unload()

adapter_config.json:   0%|          | 0.00/935 [00:00<?, ?B/s]

checkpoint-2835/adapter_model.safetensor(…):   0%|          | 0.00/5.34M [00:00<?, ?B/s]

In [9]:
# Sanity check
for i in range(11):
    print(f"Merged model layer-{i} attention value weight norm:", merged_model.roberta.encoder.layer[i].attention.self.value.weight.norm().item())
print()
print("Merged model qa_outputs weight norm:", merged_model.qa_outputs.weight.norm().item())
print("Merged model qa_outputs bias norm:", merged_model.qa_outputs.bias.norm().item())

Merged model layer-0 attention value weight norm: 28.366104125976562
Merged model layer-1 attention value weight norm: 28.84681510925293
Merged model layer-2 attention value weight norm: 30.103973388671875
Merged model layer-3 attention value weight norm: 36.346412658691406
Merged model layer-4 attention value weight norm: 38.2970085144043
Merged model layer-5 attention value weight norm: 41.631202697753906
Merged model layer-6 attention value weight norm: 38.205692291259766
Merged model layer-7 attention value weight norm: 38.959476470947266
Merged model layer-8 attention value weight norm: 39.16414260864258
Merged model layer-9 attention value weight norm: 34.90419006347656
Merged model layer-10 attention value weight norm: 31.03905487060547

Merged model qa_outputs weight norm: 1.0304267406463623
Merged model qa_outputs bias norm: 0.004130995832383633


In [10]:
# Upload the merged model to Hugging Face
merged_model.push_to_hub(hub_merged_model_id)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.push_to_hub(hub_merged_model_id)

print(f"Merged model uploaded to: https://huggingface.co/{hub_merged_model_id}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ms0fqvb/model.safetensors:   3%|2         | 32.0MB / 1.11GB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mplhlw7bw6/tokenizer.json: 100%|##########| 17.1MB / 17.1MB            

  ...6/sentencepiece.bpe.model: 100%|##########| 5.07MB / 5.07MB            

Merged model uploaded to: https://huggingface.co/alxxtexxr/XLM-R-Base-squad-en-6K-s1337-LoRA-a2m-ckpt2835-v261127123957
